# Territorial and Expenditure Analysis
## SIOPE School Spending Through Years and Regions

This notebook explores school expenditure patterns across Italian territories, analyzing regional differences, provincial distribution, and expenditure trends over time (2020-2026).

**Analysis Focus:**
- Regional expenditure distribution and growth rates
- Provincial and municipal-level spending patterns
- Budget category breakdown by territory
- Temporal evolution of school spending
- Identification of high/low spending regions and trends

## 1. Data Loading and Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Paths
SIOPE_DIR = Path("local_data/SIOPE")
PROCESSED_DIR = Path("local_data/processed")

print("Loading processed SIOPE data...")

# Load SIOPE processed files
expenditure_by_region = pd.read_csv(PROCESSED_DIR / "siope_expenditure_by_region_year.csv", index_col=0)
school_count_by_region = pd.read_csv(PROCESSED_DIR / "siope_school_count_by_region_year.csv", index_col=0)
school_expenditure_summary = pd.read_csv(PROCESSED_DIR / "siope_school_expenditure_summary.csv")
budget_category_breakdown = pd.read_csv(PROCESSED_DIR / "siope_budget_category_breakdown.csv", index_col=0)

# Load raw SIOPE data for detailed analysis
siope_uscite_files = sorted(SIOPE_DIR.glob("siope_uscite_*.csv"))
siope_dfs = []
for file in siope_uscite_files:
    year = int(file.stem.split('_')[-1])
    if year >= 2020:
        df = pd.read_csv(file, names=['codice_ente', 'anno', 'mese', 'codice_gestionale', 'importo_centesimi'])
        df['importo_centesimi'] = pd.to_numeric(df['importo_centesimi'], errors='coerce')
        df['importo_euro'] = df['importo_centesimi'] / 100
        siope_dfs.append(df)

siope_uscite = pd.concat(siope_dfs, ignore_index=True) if siope_dfs else pd.DataFrame()
siope_registry = pd.read_csv(SIOPE_DIR / "siope_anagrafiche_scuole.csv")

# Merge with territorial info
siope_merged = siope_uscite.merge(
    siope_registry[['codice_ente', 'codice_regione', 'codice_provincia', 'codice_comune', 'denominazione']],
    on='codice_ente',
    how='left'
).dropna(subset=['importo_euro'])

print(f"✓ SIOPE USCITE: {len(siope_merged):,} records")
print(f"✓ SIOPE Registry: {len(siope_registry):,} schools")
print(f"✓ Years: {sorted(siope_merged['anno'].unique())}")
print(f"✓ Regions: {siope_merged['codice_regione'].nunique()}")
print(f"✓ Total expenditure: €{siope_merged['importo_euro'].sum():,.2f}")

## 2. Regional Overview and Ranking

In [ ]:
print("="*80)
print("REGIONAL EXPENDITURE OVERVIEW - ALL YEARS (2020-2026)")
print("="*80)

# Calculate regional totals
regional_totals = siope_merged.groupby('codice_regione').agg({
    'importo_euro': ['sum', 'mean', 'count'],
    'codice_ente': 'nunique'
}).round(2)
regional_totals.columns = ['Total Expenditure (€)', 'Mean per record (€)', 'Record count', 'Unique schools']
regional_totals = regional_totals.sort_values('Total Expenditure (€)', ascending=False)

print("\nTOP 20 REGIONS BY TOTAL EXPENDITURE")
print(regional_totals.head(20))

# Calculate per-school average
regional_totals['Avg per school (€)'] = regional_totals['Total Expenditure (€)'] / regional_totals['Unique schools']
regional_totals['Avg records per school'] = regional_totals['Record count'] / regional_totals['Unique schools']

print("\n\nAVERAGE EXPENDITURE PER SCHOOL BY REGION (Top 20)")
per_school = regional_totals[['Unique schools', 'Avg per school (€)', 'Avg records per school']].head(20)
print(per_school)

# Visualization: Top regions by expenditure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

top_regions = regional_totals['Total Expenditure (€)'].nlargest(15)
colors_exp = plt.cm.RdYlGn_r(np.linspace(0.3, 0.8, len(top_regions)))
ax1.barh(range(len(top_regions)), top_regions.values / 1e6, color=colors_exp)
ax1.set_yticks(range(len(top_regions)))
ax1.set_yticklabels(top_regions.index)
ax1.set_xlabel('Total Expenditure (€ Millions)')
ax1.set_title('Top 15 Regions by Total Expenditure (2020-2026)', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Average per school
top_avg = regional_totals['Avg per school (€)'].nlargest(15)
colors_avg = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_avg)))
ax2.barh(range(len(top_avg)), top_avg.values / 1e6, color=colors_avg)
ax2.set_yticks(range(len(top_avg)))
ax2.set_yticklabels(top_avg.index)
ax2.set_xlabel('Average per School (€ Millions)')
ax2.set_title('Highest Avg Expenditure per School by Region', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Regional concentration analysis
print("\n\nREGIONAL CONCENTRATION ANALYSIS")
total_exp = regional_totals['Total Expenditure (€)'].sum()
top5_share = regional_totals.head(5)['Total Expenditure (€)'].sum() / total_exp * 100
top10_share = regional_totals.head(10)['Total Expenditure (€)'].sum() / total_exp * 100
print(f"Top 5 regions account for {top5_share:.1f}% of total expenditure")
print(f"Top 10 regions account for {top10_share:.1f}% of total expenditure")
print(f"Total regions: {len(regional_totals)}")

## 3. Regional Temporal Analysis - Growth Rates

In [ ]:
# Regional expenditure by year (from processed data)
print("REGIONAL GROWTH ANALYSIS")
print("="*80)

# Calculate year-over-year growth for top regions
regional_by_year = expenditure_by_region.copy()
top_regions_list = regional_by_year.sum(axis=0).nlargest(10).index

print(f"\nTop 10 Regions - Expenditure by Year (€ Millions)\n")
growth_analysis = regional_by_year.loc[top_regions_list].T / 1e6
growth_analysis = growth_analysis.fillna(0)
print(growth_analysis.round(2))

# Calculate growth rates
print("\n\nYEAR-OVER-YEAR GROWTH RATES (Top 10 Regions)\n")
growth_rates = growth_analysis.pct_change() * 100
print(growth_rates.round(1))

# Visualization: Regional trends for top regions
fig, ax = plt.subplots(figsize=(14, 8))
for region in top_regions_list[:8]:  # Plot top 8 for clarity
    years = growth_analysis.index.astype(int)
    ax.plot(years, growth_analysis[region], marker='o', linewidth=2, label=f'Region {region}')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Expenditure (€ Millions)', fontsize=12)
ax.set_title('Expenditure Trends - Top 8 Regions (2020-2026)', fontweight='bold', fontsize=14)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate CAGR (Compound Annual Growth Rate) 2020-2025
print("\n\nCOMPOUND ANNUAL GROWTH RATE 2020-2025 (Top 10 Regions)\n")
cagr_data = []
for region in top_regions_list:
    exp_2020 = growth_analysis.loc[2020, region] if 2020 in growth_analysis.index else None
    exp_2025 = growth_analysis.loc[2025, region] if 2025 in growth_analysis.index else None
    if exp_2020 and exp_2025 and exp_2020 > 0:
        cagr = (((exp_2025 / exp_2020) ** (1/5)) - 1) * 100
        cagr_data.append({'Region': region, 'CAGR 2020-2025 (%)': cagr})

cagr_df = pd.DataFrame(cagr_data).sort_values('CAGR 2020-2025 (%)', ascending=False)
print(cagr_df.to_string(index=False))

## 4. Provincial Analysis

In [ ]:
print("="*80)
print("PROVINCIAL EXPENDITURE ANALYSIS")
print("="*80)

# Provincial aggregation
provincial_totals = siope_merged.groupby('codice_provincia').agg({
    'importo_euro': ['sum', 'mean'],
    'codice_ente': 'nunique',
    'codice_regione': 'first'
}).round(2)
provincial_totals.columns = ['Total Expenditure (€)', 'Mean per record (€)', 'Unique schools', 'Region']
provincial_totals = provincial_totals.sort_values('Total Expenditure (€)', ascending=False)

print("\nTOP 20 PROVINCES BY TOTAL EXPENDITURE")
print(provincial_totals.head(20)[['Total Expenditure (€)', 'Unique schools', 'Region']])

# Calculate per-province metrics
provincial_totals['Exp per school'] = provincial_totals['Total Expenditure (€)'] / provincial_totals['Unique schools']

# Distribution visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Top provinces
top_prov = provincial_totals['Total Expenditure (€)'].nlargest(15)
ax1.barh(range(len(top_prov)), top_prov.values / 1e6, color='steelblue', alpha=0.7)
ax1.set_yticks(range(len(top_prov)))
ax1.set_yticklabels(top_prov.index)
ax1.set_xlabel('Expenditure (€ Millions)')
ax1.set_title('Top 15 Provinces by Expenditure', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Distribution of provinces by expenditure size
exp_ranges = ['<10M', '10-50M', '50-100M', '100-500M', '>500M']
province_dist = pd.cut(provincial_totals['Total Expenditure (€)'] / 1e6,
                       bins=[0, 10, 50, 100, 500, float('inf')],
                       labels=exp_ranges).value_counts()
colors_dist = plt.cm.Set3(range(len(province_dist)))
ax2.bar(province_dist.index, province_dist.values, color=colors_dist, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Expenditure Range')
ax2.set_ylabel('Number of Provinces')
ax2.set_title('Distribution of Provinces by Expenditure Size', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Schools per province
ax3.hist(provincial_totals['Unique schools'], bins=30, color='darkgreen', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Number of Schools')
ax3.set_ylabel('Number of Provinces')
ax3.set_title('Distribution of Schools per Province', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Expenditure per school distribution
exp_per_school = provincial_totals['Exp per school'] / 1e6
ax4.hist(exp_per_school, bins=30, color='coral', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Expenditure per School (€ Millions)')
ax4.set_ylabel('Number of Provinces')
ax4.set_title('Distribution of Avg Expenditure per School', fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Statistics
print(f"\n\nPROVINCIAL STATISTICS")
print(f"Total provinces: {len(provincial_totals)}")
print(f"Avg expenditure per province: €{provincial_totals['Total Expenditure (€)'].mean():,.2f}")
print(f"Median expenditure per province: €{provincial_totals['Total Expenditure (€)'].median():,.2f}")
print(f"Avg schools per province: {provincial_totals['Unique schools'].mean():.0f}")
print(f"Avg expenditure per school: €{provincial_totals['Exp per school'].mean():,.2f}")

## 5. Budget Category Analysis by Territory

In [ ]:
print("="*80)
print("BUDGET CATEGORY ANALYSIS")
print("="*80)

# Top budget categories overall
budget_by_category = siope_merged.groupby('codice_gestionale').agg({
    'importo_euro': 'sum',
    'codice_ente': 'nunique'
}).sort_values('importo_euro', ascending=False)
budget_by_category.columns = ['Total (€)', 'Schools']

print("\nTOP 20 BUDGET CATEGORIES (Overall)")
print(budget_by_category.head(20))

# Pie chart: Top budget categories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

top_categories = budget_by_category['Total (€)'].nlargest(10)
other_total = budget_by_category['Total (€)'].iloc[10:].sum()
pie_data = list(top_categories.values) + [other_total]
pie_labels = list(top_categories.index) + ['Other']

colors = plt.cm.Set3(np.linspace(0, 1, len(pie_data)))
wedges, texts, autotexts = ax1.pie(pie_data, labels=pie_labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Budget Category Distribution\n(Top 10 + Others)', fontweight='bold')
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontsize(9)

# Regional variation in budget categories
print("\n\nBUDGET CATEGORY VARIATION BY REGION")
print("Showing average spend per school by category (Top regions)\n")

# Get top 5 regions
top_5_regions = regional_totals.head(5).index

# For each region, show budget category composition
for region in top_5_regions:
    region_data = siope_merged[siope_merged['codice_regione'] == region]
    region_budget = region_data.groupby('codice_gestionale')['importo_euro'].sum().sort_values(ascending=False).head(5)
    print(f"\nRegion {region} - Top 5 Budget Categories:")
    for cat, amt in region_budget.items():
        print(f"  {cat}: €{amt:,.0f}")

# Stacked bar chart showing category composition for top regions
category_by_region = []
for region in top_5_regions:
    region_data = siope_merged[siope_merged['codice_regione'] == region]
    region_budget = region_data.groupby('codice_gestionale')['importo_euro'].sum().sort_values(ascending=False).head(5)
    region_budget['Region'] = region
    category_by_region.append(region_budget)

# Create visualization of category breakdown
ax2_data = siope_merged.groupby(['codice_regione', 'codice_gestionale'])['importo_euro'].sum().unstack(fill_value=0)
top_regions_cat = ax2_data.loc[top_5_regions]

top_regions_cat.plot(kind='barh', stacked=True, ax=ax2, colormap='tab20')
ax2.set_xlabel('Expenditure (€)')
ax2.set_ylabel('Region')
ax2.set_title('Budget Category Composition - Top 5 Regions', fontweight='bold')
ax2.legend(title='Budget Category', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 6. Key Insights and Summary

In [ ]:
print("\n" + "="*80)
print("TERRITORIAL AND EXPENDITURE ANALYSIS - EXECUTIVE SUMMARY")
print("="*80)

# Calculate key metrics
total_exp_all = siope_merged['importo_euro'].sum()
total_schools = siope_merged['codice_ente'].nunique()
avg_exp_per_school = total_exp_all / total_schools

# Regional concentration
top_5_exp = regional_totals.head(5)['Total Expenditure (€)'].sum()
top_5_pct = (top_5_exp / total_exp_all) * 100

# Growth calculation
exp_2020 = siope_merged[siope_merged['anno'] == 2020]['importo_euro'].sum()
exp_2025 = siope_merged[siope_merged['anno'] == 2025]['importo_euro'].sum()
growth_pct = ((exp_2025 - exp_2020) / exp_2020) * 100 if exp_2020 > 0 else 0

print(f"""
📊 OVERALL STATISTICS (2020-2026):
   • Total Expenditure: €{total_exp_all:,.2f}
   • Schools: {total_schools:,}
   • Average per School: €{avg_exp_per_school:,.2f}
   • Regions: {siope_merged['codice_regione'].nunique()}
   • Provinces: {siope_merged['codice_provincia'].nunique()}

📈 GROWTH METRICS:
   • 2020-2025 Growth: {growth_pct:+.1f}%
   • Expenditure 2020: €{exp_2020:,.2f}
   • Expenditure 2025: €{exp_2025:,.2f}

🗺️  REGIONAL CONCENTRATION:
   • Top 5 Regions Account for: {top_5_pct:.1f}% of Total Expenditure
   • Leading Region: {regional_totals.index[0]} (€{regional_totals.iloc[0]['Total Expenditure (€)']:,.2f})
   • 2nd Leading Region: {regional_totals.index[1]} (€{regional_totals.iloc[1]['Total Expenditure (€)']:,.2f})

💰 PROVINCIAL DETAILS:
   • Average Expenditure per Province: €{provincial_totals['Total Expenditure (€)'].mean():,.2f}
   • Median Expenditure per Province: €{provincial_totals['Total Expenditure (€)'].median():,.2f}
   • Average Schools per Province: {provincial_totals['Unique schools'].mean():.0f}
   • Average Expenditure per School (Provincial): €{provincial_totals['Exp per school'].mean():,.2f}

🏆 TOP PERFORMERS:
   • Highest Regional Expenditure: Region {regional_totals.index[0]}
   • Highest Provincial Expenditure: Province {provincial_totals.index[0]}
   • Most Schools in Region: Region {school_count_by_region.sum(axis=1).idxmax()}

⚠️  ANALYSIS NOTES:
   • Schools only joined SIOPE in 2020 (no data before then)
   • 2026 data is partial (updated to April 30, 2026)
   • Budget categories represent different expenditure types (personnel, materials, etc.)
   • Some regions/provinces may have missing data due to reporting delays

🔍 NEXT STEPS FOR DEEPER ANALYSIS:
   1. Correlate expenditure with student population data from Minister datasets
   2. Analyze spending efficiency (expenditure per student)
   3. Identify spending anomalies and outliers
   4. Cross-reference with school type and location characteristics
   5. Compare regional spending patterns with education outcomes
""")

print("="*80)